# 09 — Neural Inertial Odometry Network Training (TLIO-Style)

**SIH PS 26168 — Intelligent Dead Reckoning**

> **Section 18:** Lightweight TLIO-inspired Neural Inertial Odometry model predicting relative displacement, forward velocity, and uncertainty.
> **Section 2:** Folds OdoNet forward velocity estimation directly into the odometry head.
> **Section 13:** Remote GPU execution requirement (`torch.cuda.is_available() == True`).

### Objectives:
1. Train a Dilated Temporal Convolutional Network (TCN) on real IO-VNBD IMU sequence windows.
2. Jointly predict relative 2D displacement $[\Delta x, \Delta y]$, instantaneous vehicle speed $v_{\text{fwd}}$, and calibrated uncertainty $\sigma^2$.
3. Optimize via Heteroscedastic Gaussian Negative Log-Likelihood (NLL) and velocity MSE loss.
4. Save best checkpoint to `checkpoints/inertial_odometry/inertial_odometry_best.pt`.

## 1. Environment & GPU Preflight Check (Section 13)

In [ ]:
import os, sys, json, time
from pathlib import Path
import yaml
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

# Strict GPU Check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Execution Device: {device}')
if torch.cuda.is_available():
    print(f'GPU Device Name: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('[NOTICE] CUDA not detected on local host. Remote Lightning GPU execution required.')

ckpt_dir = PROJECT_ROOT / 'checkpoints' / 'inertial_odometry'
plots_dir = PROJECT_ROOT / 'plots' / 'inertial_odometry'
ckpt_dir.mkdir(parents=True, exist_ok=True)
plots_dir.mkdir(parents=True, exist_ok=True)

## 2. Load Configuration & Motion Datasets

In [ ]:
with open(PROJECT_ROOT / 'configs' / 'training.yaml', 'r') as f:
    cfg = yaml.safe_load(f)['inertial_odometry']

print('Neural Inertial Odometry Configuration:')
print(json.dumps(cfg, indent=2))

from src.datasets.inertial_odometry_dataset import InertialOdometryDataset
from src.models.inertial_odometry import NeuralInertialOdometry

train_ds = InertialOdometryDataset(
    split='train',
    window_size=100,
    stride=20
)

val_ds = InertialOdometryDataset(
    split='val',
    window_size=100,
    stride=30
)

batch_size = cfg.get('batch_size', 64)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

print(f'Train Batches: {len(train_loader)} | Validation Batches: {len(val_loader)}')

## 3. Instantiate Neural Inertial Odometry Model

In [ ]:
model = NeuralInertialOdometry(
    input_dim=6,
    tcn_channels=cfg.get('tcn_channels', [64, 128, 256]),
    kernel_size=cfg.get('tcn_kernel_size', 3),
    dropout=cfg.get('dropout', 0.1),
    vel_loss_weight=cfg.get('velocity_loss_weight', 0.5),
    uncertainty_weight=cfg.get('uncertainty_loss_weight', 0.1)
).to(device)

param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Neural Inertial Odometry Trainable Parameters: {param_count:,}')

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=float(cfg.get('learning_rate', 1.0e-3)),
    weight_decay=float(cfg.get('weight_decay', 1.0e-4))
)

num_epochs = cfg.get('num_epochs', 50)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-5)

## 4. Multi-Task Training Loop with Heteroscedastic NLL Loss

In [ ]:
train_losses = []
val_losses = []
val_disp_rmses = []
val_vel_rmses = []
best_val_loss = float('inf')
best_ckpt_path = ckpt_dir / 'inertial_odometry_best.pt'

scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))
print(f'Starting Neural Inertial Odometry training for {num_epochs} epochs...')

for epoch in range(1, num_epochs + 1):
    model.train()
    total_loss = 0.0
    n_train = 0
    
    for batch in train_loader:
        x_imu = batch['imu'].to(device)
        gt_disp = batch['disp'].to(device)
        gt_vel = batch['vel'].to(device)
        
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            p_disp, p_vel, p_logvar = model(x_imu)
            loss, _, _ = model.compute_loss(p_disp, p_vel, p_logvar, gt_disp, gt_vel)
            
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        n_train += 1
        
    scheduler.step()
    avg_train_loss = total_loss / max(1, n_train)
    train_losses.append(avg_train_loss)
    
    # Validation Evaluation
    model.eval()
    total_val_loss = 0.0
    disp_errors = []
    vel_errors = []
    n_val = 0
    
    with torch.no_grad():
        for batch in val_loader:
            x_imu = batch['imu'].to(device)
            gt_disp = batch['disp'].to(device)
            gt_vel = batch['vel'].to(device)
            
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                p_disp, p_vel, p_logvar = model(x_imu)
                v_loss, _, _ = model.compute_loss(p_disp, p_vel, p_logvar, gt_disp, gt_vel)
                
            total_val_loss += v_loss.item()
            n_val += 1
            
            disp_err = torch.norm(p_disp - gt_disp, dim=1).cpu().numpy()
            vel_err = torch.abs(p_vel[:, 0] - gt_vel[:, 0]).cpu().numpy()
            disp_errors.extend(disp_err)
            vel_errors.extend(vel_err)
            
    avg_val_loss = total_val_loss / max(1, n_val)
    disp_rmse = float(np.sqrt(np.mean(np.array(disp_errors) ** 2)))
    vel_rmse = float(np.sqrt(np.mean(np.array(vel_errors) ** 2)))
    
    val_losses.append(avg_val_loss)
    val_disp_rmses.append(disp_rmse)
    val_vel_rmses.append(vel_rmse)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_loss': best_val_loss,
            'disp_rmse': disp_rmse,
            'vel_rmse': vel_rmse,
            'config': cfg,
            'param_count': param_count
        }, best_ckpt_path)
        star = ' * [CHECKPOINT SAVED]'
    else:
        star = ''
        
    if epoch % 5 == 0 or epoch == 1 or star:
        print(f'Epoch [{epoch:3d}/{num_epochs:3d}] | Loss: {avg_train_loss:.4f} (Val: {avg_val_loss:.4f}) | Disp RMSE: {disp_rmse:.2f}m | Vel RMSE: {vel_rmse:.2f}m/s{star}')

print(f'Training complete! Best Validation Loss: {best_val_loss:.4f}')
print(f'Best Model Checkpoint: {best_ckpt_path}')

## 5. Learning Curves Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(range(1, len(train_losses) + 1), train_losses, 'b-', label='Train Loss (NLL + MSE)')
axes[0].plot(range(1, len(val_losses) + 1), val_losses, 'r--', label='Validation Loss')
axes[0].set_xlabel('Epoch', fontweight='bold')
axes[0].set_ylabel('Total Multi-Task Loss', fontweight='bold')
axes[0].set_title('Neural Inertial Odometry Loss Convergence', fontsize=12, fontweight='bold')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.4)

axes[1].plot(range(1, len(val_disp_rmses) + 1), val_disp_rmses, color='forestgreen', label='Displacement RMSE (m)')
axes[1].plot(range(1, len(val_vel_rmses) + 1), val_vel_rmses, color='darkorange', linestyle='--', label='Velocity RMSE (m/s)')
axes[1].set_xlabel('Epoch', fontweight='bold')
axes[1].set_ylabel('Metric Error', fontweight='bold')
axes[1].set_title('Validation Error Tracking', fontsize=12, fontweight='bold')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.4)

plt.tight_layout()
curve_path = plots_dir / 'inertial_odometry_training_curve.png'
plt.savefig(curve_path, dpi=200)
plt.close()
print(f'Training curve saved: {curve_path}')

# Summary metrics
summary = {
    'model': 'Neural Inertial Odometry (TLIO-style)',
    'epochs_trained': len(train_losses),
    'best_val_loss': round(float(best_val_loss), 4),
    'val_displacement_rmse_m': round(float(val_disp_rmses[-1]), 3),
    'val_velocity_rmse_mps': round(float(val_vel_rmses[-1]), 3),
    'trainable_params': param_count,
    'checkpoint': str(best_ckpt_path.relative_to(PROJECT_ROOT))
}
with open(PROJECT_ROOT / 'results' / 'inertial_odometry_training_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('Training summary saved to results/inertial_odometry_training_summary.json.')